# Sample 20% of annotated label data

Samples 20% of rows from every CSV under `data/annotations/label/` and writes them to `data/annotations/label_sample_20/`, mirroring the original folder structure.

Important: the **same row indices** are sampled across all annotator folders (GPT-5.2, Gemini 3.5 flash, Sonet 4.6) so the sampled comments are aligned for agreement analysis.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
FRACTION = 0.20

ROOT = Path.cwd().parents[1]  # repo root (notebook lives in src/validation/)
SRC_DIR = ROOT / "data" / "annotations" / "label"
DST_DIR = ROOT / "data" / "annotations" / "label_sample_20"

print(f"Source: {SRC_DIR}")
print(f"Destination: {DST_DIR}")

In [ ]:
csv_paths = sorted(SRC_DIR.rglob("*.csv"))
print(f"Found {len(csv_paths)} CSV files:")
for p in csv_paths:
    print(f"  {p.relative_to(SRC_DIR)}")

## Sample and save

To keep rows aligned across annotators, sampled indices are computed once per *dataset file* (the path relative to the annotator folder, e.g. `IDS/Regular_video_comments_Annotated.csv`) and reused for every annotator.

In [ ]:
rng = np.random.default_rng(SEED)
sampled_indices = {}  # dataset-relative path -> sampled row indices
summary = []

for csv_path in csv_paths:
    rel = csv_path.relative_to(SRC_DIR)          # e.g. GPT-5.2/IDS/file.csv
    dataset_key = Path(*rel.parts[1:])            # e.g. IDS/file.csv (drop annotator)

    df = pd.read_csv(csv_path)

    if dataset_key not in sampled_indices:
        n_sample = max(1, round(len(df) * FRACTION))
        idx = np.sort(rng.choice(len(df), size=n_sample, replace=False))
        sampled_indices[dataset_key] = idx

    idx = sampled_indices[dataset_key]
    if idx.max() >= len(df):
        raise ValueError(
            f"{rel}: has {len(df)} rows but another annotator's copy has more — "
            "files are not aligned across annotators."
        )

    sample = df.iloc[idx]

    out_path = DST_DIR / rel
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sample.to_csv(out_path, index=False)

    summary.append({
        "file": str(rel),
        "total_rows": len(df),
        "sampled_rows": len(sample),
    })

summary_df = pd.DataFrame(summary)
summary_df

## Verify

Check the output mirrors the source structure and that sampled rows are aligned across annotators.

In [ ]:
out_paths = sorted(DST_DIR.rglob("*.csv"))
assert {p.relative_to(DST_DIR) for p in out_paths} == {p.relative_to(SRC_DIR) for p in csv_paths}
print(f"OK — {len(out_paths)} files written, structure matches.\n")

# Spot-check alignment: same comments sampled for each annotator
# (compare with whitespace normalized — a couple of source rows differ only in newlines)
def norm(series):
    return [" ".join(str(c).split()) for c in series]

annotators = sorted({p.relative_to(SRC_DIR).parts[0] for p in csv_paths})
for dataset_key in sampled_indices:
    comments = [
        norm(pd.read_csv(DST_DIR / ann / dataset_key)["comment"])
        for ann in annotators
        if (DST_DIR / ann / dataset_key).exists()
    ]
    assert all(c == comments[0] for c in comments), f"Misaligned: {dataset_key}"
print("OK — sampled comments aligned across all annotators.")